# ⚡ 新能源行业智能体
## 一键部署 · Colab Free T4 GPU

### 🚀 使用方法
1. 先配置 Secrets: 侧边栏 **🔑 密钥** → 添加 `HF_TOKEN` 和 `NGROK_TOKEN`（都是可选的）
2. 点击下方 **▶️ 运行** 按钮
3. 首次会弹出 Google Drive 授权窗口，点击允许
4. 等待 5-8 分钟后，Cell 输出区会显示访问链接

💾 **断开重连后重跑此 Cell，模型/缓存都不丢，启动越来越快**


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ⚡ 新能源行业智能体 · 一键运行
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, sys, subprocess, time, glob

# ═══════════════════════════════════════════════
# Step 1: 读取 Secrets
# ═══════════════════════════════════════════════
print("🔐 读取 Secrets...")
try:
    from google.colab import userdata
    for name in ["HF_TOKEN", "NGROK_TOKEN"]:
        try:
            val = userdata.get(name)
            if val:
                os.environ[name] = val
                print(f"   ✅ {name}")
            else:
                print(f"   ⚠️  {name} 未配置 (可选)")
        except:
            print(f"   ⚠️  {name} 未配置 (可选)")
except:
    print("   非 Colab 环境，跳过")

# ═══════════════════════════════════════════════
# Step 2: 挂载 Google Drive
# ═══════════════════════════════════════════════
print("\n📂 挂载 Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

# 定义所有持久化目录
DRIVE_DIRS = {
    "HF_HOME": "/content/drive/MyDrive/hf_cache",
    "HF_HUB_CACHE": "/content/drive/MyDrive/hf_cache",
    "PIP_CACHE_DIR": "/content/drive/MyDrive/pip_cache",
    "VLLM_CACHE_DIR": "/content/drive/MyDrive/vllm_cache",
    "VLLM_CONFIG_ROOT": "/content/drive/MyDrive/vllm_cache",
    "NEW_ENERGY_DATA_DIR": "/content/drive/MyDrive/new-energy-data",
}
for key, path in DRIVE_DIRS.items():
    os.makedirs(path, exist_ok=True)
    os.environ[key] = path

print("   ✅ 所有缓存目录已就绪 (Google Drive)")

# 检查电价缓存
db_path = os.path.join(DRIVE_DIRS["NEW_ENERGY_DATA_DIR"], "electricity_cache.db")
if os.path.exists(db_path):
    import sqlite3
    count = sqlite3.connect(db_path).execute("SELECT COUNT(*) FROM electricity_prices").fetchone()[0]
    print(f"   📊 电价缓存: {count} 条记录 (历史会话)")

# ═══════════════════════════════════════════════
# Step 3: 安装依赖
# ═══════════════════════════════════════════════
print("\n📦 安装依赖...")
!pip install -q vllm gradio plotly pandas duckduckgo_search pyngrok huggingface_hub openai httpx requests 2>&1 | tail -1
print("   ✅ 依赖安装完成")

# ═══════════════════════════════════════════════
# Step 4: 下载/加载模型
# ═══════════════════════════════════════════════
print("\n🤖 加载模型...")
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct-AWQ"
HF_CACHE = DRIVE_DIRS["HF_HOME"]

from huggingface_hub import snapshot_download

def get_cached_path(repo_id, cache_dir):
    """检测模型是否已缓存"""
    try:
        from huggingface_hub import scan_cache_dir
        for repo in scan_cache_dir(cache_dir=cache_dir).repos:
            if repo.repo_id == repo_id and repo.revisions:
                for rev in repo.revisions:
                    if rev.last_modified:
                        p = os.path.join(cache_dir, "hub", repo.repo_path, "snapshots", rev.commit_hash)
                        if os.path.isdir(p) and os.listdir(p):
                            return p
    except:
        pass
    # 回退: 直接找目录
    hub_dir = os.path.join(cache_dir, "hub")
    if os.path.isdir(hub_dir):
        dirs = glob.glob(os.path.join(hub_dir, "models--Qwen*", "snapshots", "*"))
        for d in dirs:
            if os.path.isdir(d) and os.listdir(d):
                return d
    return None

model_path = get_cached_path(MODEL_ID, HF_CACHE)

if model_path:
    total_gb = sum(os.path.getsize(os.path.join(dp, f))
                   for dp, _, files in os.walk(model_path) for f in files) / 1e9
    print(f"   ✅ 命中缓存: {model_path} ({total_gb:.1f} GB)")
else:
    print(f"   📥 首次下载模型 (~2.5GB, 约 2-5 分钟)...")
    start = time.time()
    model_path = snapshot_download(MODEL_ID, cache_dir=HF_CACHE, resume_download=True, max_workers=4)
    print(f"   ✅ 下载完成 ({time.time()-start:.0f}秒)")

# ═══════════════════════════════════════════════
# Step 5: 验证 GPU
# ═══════════════════════════════════════════════
print("\n🔍 检查 GPU...")
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
if not gpu:
    print("   ❌ 无 GPU！运行时 → 更改运行时类型 → T4 GPU → 重跑")
    raise SystemExit(1)
print(f"   ✅ {gpu[0]}")

# ═══════════════════════════════════════════════
# Step 6: 启动 vLLM
# ═══════════════════════════════════════════════
print("\n🚀 启动 vLLM 推理引擎 (首次编译 CUDA kernel 约 2-3 分钟)...")

!pkill -f "vllm.entrypoints" 2>/dev/null || true
time.sleep(1)

log_file = "/tmp/vllm_run.log"
with open(log_file, "w") as f:
    proc = subprocess.Popen([
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--quantization", "awq",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--port", "8000",
        "--host", "0.0.0.0",
    ], stdout=f, stderr=f)

print(f"   PID: {proc.pid}")

# 等待 vLLM 就绪 (最多 5 分钟)
print("   ⏳ 等待就绪...")
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="local")
ready = False
for i in range(60):
    time.sleep(5)
    try:
        client.models.list()
        ready = True
        break
    except:
        if i % 6 == 0:
            print(f"   ...{i*5}s  (查看日志: tail -10 {log_file})")

if not ready:
    print("\n❌ vLLM 启动失败! 诊断日志:")
    !tail -40 {log_file}
    raise SystemExit(1)

print("   ✅ vLLM 已就绪!")

# ═══════════════════════════════════════════════
# Step 7: 克隆代码 + 启动 Agent
# ═══════════════════════════════════════════════
print("\n📥 拉取代码...")
REPO_DIR = "/content/new-energy-agent"
if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull -q 2>&1 | tail -1
else:
    !git clone -q https://github.com/pai-pixel/new-energy-agent.git {REPO_DIR}
    %cd {REPO_DIR}

sys.path.insert(0, REPO_DIR)
%cd {REPO_DIR}

import gradio as gr
from src.agent import NewEnergyAgent, create_ui, start_ngrok

# ngrok
print("\n🔗 启动 ngrok...")
from src.config import NGROK_TOKEN
ngrok_url = start_ngrok(7860) if NGROK_TOKEN else None

# Agent
agent = NewEnergyAgent()
demo = create_ui(agent)

# 提示信息
print()
print("=" * 60)
print("  ⚡ 新能源行业智能体 · 启动完成!")
print("=" * 60)
print()
if ngrok_url:
    print(f"  🔗 ngrok: {ngrok_url}")
print("  🌐 公网地址见下方 Cell 输出 (****.gradio.live)")
print()
print("  ⚠️  断开重连: 重新运行此 Cell 即可")
print("  ■  停止: Colab 顶部 ■ 按钮")
print()

# 自动打开 ngrok 链接
if ngrok_url:
    from IPython.display import display, Javascript
    display(Javascript(f'window.open("{ngrok_url}", "_blank");'))

# 启动 Gradio (阻塞)
demo.queue(max_size=32).launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    show_error=True,
    css=".gradio-container{max-width:900px!important}",
    theme=gr.themes.Soft(primary_hue="green"),
)

In [ ]:
# ── Cell 2: 保活 + 缓存统计 (可选) ──
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
  console.log('Colab 保活: '+new Date());
  document.querySelector('colab-connect-button').click();
}
setInterval(ClickConnect,60000);
'''))

import os, sqlite3
db = os.path.join('/content/drive/MyDrive/new-energy-data','electricity_cache.db')
if os.path.exists(db):
  c = sqlite3.connect(db)
  t = c.execute('SELECT COUNT(*) FROM electricity_prices').fetchone()[0]
  ps = c.execute('SELECT DISTINCT province FROM electricity_prices').fetchall()
  c.close()
  print(f'📊 电价缓存: {t} 条 | 省份: {", ".join(p[0] for p in ps)}')

print('✅ 保活脚本已启动')
print('💾 所有缓存均在 Google Drive，断开不丢失')

---
## 📖 使用说明

**电价查询:** 「上海上网电价」「江苏脱硫煤电价」「浙江工商业电价」

**天气查询:** 「北京天气怎么样」

**知识咨询:** 「光伏补贴政策」「碳中和是什么意思」

**多轮上下文 (自动继承):**
- 「上海上网电价」→ 「江苏呢」(自动继承上网电价)
- 「工商业电价呢」(自动继承江苏)
- 「那天气呢」(自动继承江苏天气)

> ⚡ [GitHub](https://github.com/pai-pixel/new-energy-agent)